# Deploy a Qwen3.5-397B-A17B-FP8 on SageMaker Endpoint using AWS Python API (boto3)



### [Qwen3.5](https://huggingface.co/Qwen/Qwen3.5-397B-A17B) is a multimodal mixture-of-experts model featuring a gated delta networks architecture with 397B total parameters and 17B active parameters.

#### Qwen3.5 Highlights

**Qwen3.5 features the following enhancement**:

- ***Unified Vision-Language Foundation***: Early fusion training on multimodal tokens achieves cross-generational parity with Qwen3 and outperforms Qwen3-VL models across reasoning, coding, agents, and visual understanding benchmarks.

- ***Efficient Hybrid Architecture***: Gated Delta Networks combined with sparse Mixture-of-Experts deliver high-throughput inference with minimal latency and cost overhead.

- ***Scalable RL Generalization***: Reinforcement learning scaled across million-agent environments with progressively complex task distributions for robust real-world adaptability.

- ***Global Linguistic Coverage***: Expanded support to 201 languages and dialects, enabling inclusive, worldwide deployment with nuanced cultural and regional understanding.

- ***Next-Generation Training Infrastructure***: Near-100% multimodal training efficiency compared to text-only training and asynchronous RL frameworks supporting massive-scale agent scaffolds and environment orchestration.


Please refer to official [blog](https://qwen.ai/blog?id=qwen3.5) for more details



In [ ]:
%pip install --upgrade --quiet --no-warn-conflicts boto3

In [ ]:
import time
import re
import json
import boto3
from IPython.display import display, Markdown, clear_output

boto_session = boto3.Session()
region = boto_session.region_name

sm = boto3.client("sagemaker")  # client to intreract with SageMaker
sm_runtime = boto3.client("sagemaker-runtime")  # client to intreract with SageMaker Endpoints

In [ ]:
#
# Helper functions to remove dependency on SageMaker Python SDK
#
def get_sagemaker_role():
    sts = boto3.client('sts')
    response = sts.get_caller_identity()
    assumed_role = response['Arn']
    role = re.sub(r"^(.+)sts::(\d+):assumed-role/(.+?)/.*$", r"\1iam::\2:role/\3", assumed_role)
    return role


def wait_for_endpoint(endpoint_name: str, sleep_time: int=60):
    ind = "."
    progress = f"Waiting for '{endpoint_name}': "
    print(progress)
    
    status = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
    
    while status == "Creating":
        time.sleep(sleep_time)
        
        status = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
        
        clear_output(wait=True)
        progress += ind
        print(progress)
  
    print(f"Endpoint: '{endpoint_name}', Status: '{status}'")


def wait_for_ic(ic_name: str, sleep_time: int=60):
    ind = "."
    progress = f"Waiting for '{ic_name}': "
    print(progress)
    
    status = sm.describe_inference_component(InferenceComponentName=ic_name)["InferenceComponentStatus"]
    
    while status == "Creating":
        time.sleep(sleep_time)
        
        status = sm.describe_inference_component(InferenceComponentName=ic_name)["InferenceComponentStatus"]
        
        clear_output(wait=True)
        progress += ind
        print(progress)
  
    print(f"IC: '{ic_name}', Status: '{status}'")

In [ ]:
#
# Overwrite with your role ARN if you are running this notebook outside of SageMaker Studio
#
role = None

if role == None:
    role = get_sagemaker_role()
print(role)

## Container

In [ ]:
instance = {"type": "ml.g7e.48xlarge", "num_gpu": 8}
model_id = "Qwen/Qwen3.5-397B-A17B-FP8"
model_name = f"model-{time.strftime('%y%m%d-%H%M%S')}"
endpoint_name = model_name
endpoint_config_name = model_name
timeout = 1200
variant_name = "v1"

##### Amazon SageMaker AI makes model deployment process very easy. The deployment steps are exactly the same regardless of model framework container you choose. Below we show you 3 options:
1. DLC vLLM (0.17.1)
2. DLC SGLang (0.5.9)
3. SageMaker LMI (v22)

#### Please chose one option and run only the cell for your container of choice

#### We will be using native Qwen3.5 multi-token predictions (MTP). The net effect is higher throughput (more tokens per second) with no degradation in output quality, since the verification step ensures the final output matches what standard autoregressive decoding would produce. It's essentially "free" speculation baked into the model itself, removing the overhead of training and serving a separate smaller draft model.

### Option 1: vLLM

In [ ]:
inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/vllm:0.17.1-gpu-py312-cu129-ubuntu22.04-sagemaker"

spec_config = {
    "method": "mtp",
    "num_speculative_tokens": 3
}

common_env = {
    "HF_MODEL_ID": model_id,
}

vllm_env = {
    "SM_VLLM_MODEL": model_id,
    "SM_VLLM_TENSOR_PARALLEL_SIZE": json.dumps(instance["num_gpu"]),
    "SM_VLLM_MAX_MODEL_LEN": "32768",
    "SM_VLLM_ENABLE_AUTO_TOOL_CHOICE": "true",
    "SM_VLLM_TOOL_CALL_PARSER": "qwen3_coder",
    "SM_VLLM_REASONING_PARSER": "qwen3",
    "SM_VLLM_SPECULATIVE_CONFIG": json.dumps(spec_config)
}
env = common_env | vllm_env

### Option 2: SGLang

In [ ]:
inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/sglang:0.5.9-gpu-py312-cu129-ubuntu24.04-sagemaker"

common_env = {
    "SGLANG_DISABLE_CUDNN_CHECK": "1",
    "SM_SGLANG_ATTENTION_BACKEND": "triton",
    "SM_SGLANG_MEM_FRACTION_STATIC": "0.8",
    "SM_SGLANG_FP8_GEMM_BACKEND": "triton",
    "SM_SGLANG_MOE_RUNNER_BACKEND": "triton",
    "SM_SGLANG_MODEL_PATH": model_id
}
sgl_env = {
    "SM_SGLANG_TENSOR_PARALLEL_SIZE": json.dumps(instance["num_gpu"]),
    "SM_SGLANG_CONTEXT_LENGTH": "32768",
    "SM_SGLANG_TOOL_CALL_PARSER": "qwen3_coder",
    "SM_SGLANG_REASONING_PARSER": "qwen3",
    "SM_SGLANG_SPECULATIVE_ALGORITHM": "EAGLE",
    "SM_SGLANG_SPECULATIVE_NUM_STEPS": "3",
    "SM_SGLANG_SPECULATIVE_EAGLE_TOPK": "1",
    "SM_SGLANG_SPECULATIVE_NUM_DRAFT_TOKENS": "4",
    "SM_SGLANG_ENABLE_FLASHINFER_ALLREDUCE_FUSION": "true",
}
env = common_env | sgl_env

### Option 3: LMI

In [ ]:
CONTAINER_VERSION = "0.36.0-lmi22.0.0-cu129"
inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/djl-inference:{CONTAINER_VERSION}"

spec_config = {
    "method": "mtp",
    "num_speculative_tokens": 3
}
common_env = {
    "HF_MODEL_ID": model_id,
}
lmi_env = {
    "SERVING_FAIL_FAST": "true",
    "OPTION_ASYNC_MODE": "true",
    "OPTION_TENSOR_PARALLEL_DEGREE": json.dumps(instance["num_gpu"]),
    "OPTION_MAX_MODEL_LEN": "32768",
    "OPTION_ENABLE_AUTO_TOOL_CHOICE": "true",
    "OPTION_TOOL_CALL_PARSER": "qwen3_coder",
    "OPTION_REASONING_PARSER": "qwen3",
    "OPTION_SPECULATIVE_CONFIG": json.dumps(spec_config)
}
env = common_env | lmi_env

## Deployment

In [ ]:
_ = sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={
        "Image": inference_image,
        "Environment": env,
    },
)

In [ ]:
_ = sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[
        {
            "VariantName": variant_name,
            "ModelName": model_name,
            "InstanceType": instance["type"],
            "InitialInstanceCount": 1,
            "ContainerStartupHealthCheckTimeoutInSeconds": timeout,
        },
    ],
)

_ = sm.create_endpoint(EndpointName=endpoint_name, 
                       EndpointConfigName=endpoint_config_name)

_ = wait_for_endpoint(endpoint_name)

---
## Inference examples

### Best Practices

To achieve optimal performance, we recommend the following settings:

#### Sampling Parameters:

We suggest using the following sets of sampling parameters depending on the mode and task type:

- **Thinking mode for general tasks:** `temperature=1.0, top_p=0.95, top_k=20, min_p=0.0, presence_penalty=1.5, repetition_penalty=1.0`
- **Thinking mode for precise coding tasks (e.g., WebDev):** `temperature=0.6, top_p=0.95, top_k=20, min_p=0.0, presence_penalty=0.0, repetition_penalty=1.0`
- **Instruct (or non-thinking) mode for general tasks:** `temperature=0.7, top_p=0.8, top_k=20, min_p=0.0, presence_penalty=1.5, repetition_penalty=1.0`
- **Instruct (or non-thinking) mode for reasoning tasks:** `temperature=1.0, top_p=1.0, top_k=40, min_p=0.0, presence_penalty=2.0, repetition_penalty=1.0`

For supported frameworks, you can adjust the `presence_penalty` parameter between 0 and 2 to reduce endless repetitions. However, using a higher value may occasionally result in language mixing and a slight decrease in model performance.

#### Adequate Output Length

We recommend using an output length of 32,768 tokens for most queries. For benchmarking on highly complex problems, such as those found in math and programming competitions, we suggest setting the max output length to 81,920 tokens. This provides the model with sufficient space to generate detailed and comprehensive responses, thereby enhancing its overall performance.

#### Standardize Output Format

We recommend using prompts to standardize model outputs when benchmarking.

- **Math Problems:** Include `"Please reason step by step, and put your final answer within \boxed{}."` in the prompt.
- **Multiple-Choice Questions:** Add the following JSON structure to the prompt to standardize responses: `"Please show your choice in the answer field with only the choice letter, e.g., "answer": "C"."`

#### No Thinking Content in History

In multi-turn conversations, the historical model output should only include the final output part and does not need to include the thinking content. It is implemented in the provided chat template in Jinja2. However, for frameworks that do not directly use the Jinja2 chat template, it is up to the developers to ensure that the best practice is followed.

#### Long Video Understanding

To optimize inference efficiency for plain text and images, the `size` parameter in the released `video_preprocessor_config.json` is conservatively configured. It is recommended to set the `longest_edge` parameter in the video_preprocessor_config file to `469762048` (corresponding to 224k video tokens) to enable higher frame-rate sampling for hour-scale videos and thereby achieve superior performance. For example,

```json
{"longest_edge": 469762048, "shortest_edge": 4096}
```

Alternatively, override the default values via engine startup parameters. For implementation details, refer to: [vLLM](https://huggingface.co/Qwen/Qwen3.5-4B) / [SGLang](https://huggingface.co/Qwen/Qwen3.5-4B).

---
#### Text inference

In [10]:
payload={
    "messages": [
        {"role": "user", "content": "Name popular places to visit in London?"}
    ],
}

start_time = time.time()
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                 Body=json.dumps(payload),
                                 ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
end_time = time.time()

print(f"✅ Response time: {end_time-start_time:.2f}s\n")
display(Markdown(response["choices"][0]["message"]["content"]))

usage = response["usage"] 
print(f'-----------------------\n{usage}')

✅ Response time: 16.09s





London is massive and packed with history, culture, and modern attractions. To help you plan, here are the most popular places to visit, categorized by interest:

### 1. The Iconic Landmarks (The "Must-Sees")
*   **The Tower of London:** A historic castle on the Thames. Home to the **Crown Jewels**, famous Beefeater guards, and centuries of dark history.
*   **Big Ben & Houses of Parliament:** The most recognizable clock tower in the world. Best viewed from Westminster Bridge.
*   **Westminster Abbey:** The setting for every Royal coronation since 1066 and many royal weddings.
*   **Tower Bridge:** The iconic bascule bridge (often mistaken for London Bridge). You can walk across the high-level walkways for a view.
*   **Buckingham Palace:** The King's official residence. Don't miss the **Changing of the Guard** ceremony (check the schedule in advance).
*   **The London Eye:** A giant Ferris wheel on the South Bank offering panoramic views of the city.

### 2. World-Class Museums & Galleries
*Note: Most major museums in London are **free** to enter.*
*   **The British Museum:** Home to the Rosetta Stone, Elgin Marbles, and mummies. It covers human history from beginning to present.
*   **Natural History Museum:** Famous for its dinosaur skeletons, blue whale model, and stunning architecture. Great for families.
*   **Victoria and Albert Museum (V&A):** The world's greatest museum of art, design, and performance.
*   **Tate Modern:** Housed in a former power station, this is the premier gallery for international modern art.
*   **The National Gallery:** Located in Trafalgar Square, featuring European painting from the 13th to the 19th centuries.

### 3. Markets & Shopping
*   **Borough Market:** London's oldest food market. Perfect for lunch, gourmet street food, and fresh produce.
*   **Covent Garden:** Known for its apple market, street performers, boutique shops, and the Royal Opera House.
*   **Camden Market:** A hub for alternative culture, vintage clothing, street food, and quirky crafts.
*   **Portobello Road Market:** Located in Notting Hill, this is the world's largest antique market (best on Saturdays).
*   **Harrods:** The legendary luxury department store in Knightsbridge. Even if you don't buy anything, the Food Hall is worth seeing.

### 4. Royal Parks & Green Spaces
*   **Hyde Park:** One of the largest parks in London. Great for boating on the Serpentine or visiting the Diana Memorial Fountain.
*   **St James's Park:** The most "royal" park, located between Buckingham Palace and Westminster. Famous for its pelicans.
*   **Regent's Park:** Home to **London Zoo** and beautiful rose gardens.
*   **Kensington Gardens:** Adjacent to Hyde Park, home to Kensington Palace and the Peter Pan statue.

### 5. Best Views of the City
*   **The Shard:** The tallest building in the UK. The viewing gallery offers the highest view in Western Europe.
*   **Sky Garden:** Located at "20 Fenchurch Street" (the Walkie Talkie building). It offers great views and is **free**, but you must book a time slot in advance.
*   **Primrose Hill:** A natural hill in Regent's Park that offers a classic, free panoramic view of the skyline.

### 6. Trendy Neighborhoods to Explore
*   **Soho:** The heart of London's nightlife, dining, and entertainment district.
*   **Shoreditch:** The center of East London's cool scene, known for street art, vintage shops, and tech startups.
*   **Notting Hill:** Famous for its colorful pastel houses and the annual Carnival.
*   **South Bank:** A walkway along the Thames filled with book markets, restaurants, and culture, stretching from the London Eye to Tower Bridge.

### 💡 Practical Tips for Visiting London
*   **Transport:** Use the **Tube (Underground)**. You do not need to buy paper tickets; simply tap your contactless credit card, Apple Pay, or Google Pay on the yellow readers.
*   **Book Ahead:** For paid attractions (Tower of London, London Eye, The Shard), book tickets online weeks in advance to save money and guarantee entry.
*   **Walking:** London is a walking city. Some of the best experiences happen just wandering between neighborhoods.
*   **Weather:** It can rain at any time of year. Always carry a light rain jacket or umbrella.

**Suggested 3-Day Mini-Itinerary:**
*   **Day 1:** Westminster (Big Ben, Abbey, Buckingham Palace) & South Bank.
*   **Day 2:** The City (Tower of London, Tower Bridge, St Paul's) & Borough Market.
*   **Day 3:** Museums (South Kensington) & Shopping (Covent Garden/Soho).

-----------------------
{'prompt_tokens': 18, 'total_tokens': 2111, 'completion_tokens': 2093, 'prompt_tokens_details': None}


---
#### Extracting text from image

![image](https://ofasys-multimodal-wlcb-3-toshanghai.oss-accelerate.aliyuncs.com/wpf272043/keepme/image/receipt.png)

In [11]:
payload = {
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": "https://ofasys-multimodal-wlcb-3-toshanghai.oss-accelerate.aliyuncs.com/wpf272043/keepme/image/receipt.png"
                    }
                },
                {
                    "type": "text",
                    "text": "Read all the text in the image."
                }
            ]
        }
    ],
}

start_time = time.time()
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                 Body=json.dumps(payload),
                                 ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
end_time = time.time()

print(f"✅ Response time: {end_time-start_time:.2f}s\n")
display(Markdown(response["choices"][0]["message"]["content"]))

usage = response["usage"] 
print(f'-----------------------\n{usage}')

✅ Response time: 8.38s





Auntie Anne's
CINNAMON SUGAR
1 x 17,000
17,000
SUB TOTAL
17,000
GRAND TOTAL
17,000
CASH IDR
20,000
CHANGE DUE
3,000

-----------------------
{'prompt_tokens': 249, 'total_tokens': 822, 'completion_tokens': 573, 'prompt_tokens_details': None}


---
#### Image Reasoning

![math](https://qianwen-res.oss-accelerate.aliyuncs.com/Qwen3.5/demo/CI_Demo/mathv-1327.jpg)

In [12]:
payload = {
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": "https://qianwen-res.oss-accelerate.aliyuncs.com/Qwen3.5/demo/CI_Demo/mathv-1327.jpg"
                    }
                },
                {
                    "type": "text",
                    "text": "The centres of the four illustrated circles are in the corners of the square. The two big circles touch each other and also the two little circles. With which factor do you have to multiply the radii of the little circles to obtain the radius of the big circles?\nChoices:\n(A) $\\frac{2}{9}$\n(B) $\\sqrt{5}$\n(C) $0.8 \\cdot \\pi$\n(D) 2.5\n(E) $1+\\sqrt{2}$"
                }
            ]
        }
    ],
}

start_time = time.time()
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                 Body=json.dumps(payload),
                                 ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
end_time = time.time()

print(f"✅ Response time: {end_time-start_time:.2f}s\n")
display(Markdown(response["choices"][0]["message"]["content"]))

usage = response["usage"] 
print(f'-----------------------\n{usage}')

✅ Response time: 115.44s





The problem asks for the factor by which we must multiply the radius of the little circles ($r$) to get the radius of the big circles ($R$). In other words, we need to find the ratio $R/r$.

**1. Define the Geometry**
*   Let the side length of the square be $a$.
*   The centers of the four circles are at the corners of the square.
*   From the image and description, we have two big circles (radius $R$) and two little circles (radius $r$).
*   The arrangement is alternating: a big circle and a little circle are at adjacent corners. For example, if the bottom-left corner has a big circle, the top-left and bottom-right corners have little circles, and the top-right corner has a big circle.
    *   This is confirmed by the condition "The two big circles touch each other". This implies they are at opposite corners (diagonal), because if they were at adjacent corners, they would touch along the side, and the condition about touching the little circles would be harder to satisfy simultaneously with the visual. Also, visually, the big circles are at bottom-left and top-right.
    *   So, Big Circles are at opposite corners (distance between centers = diagonal = $a\sqrt{2}$).
    *   Little Circles are at the other opposite corners.
    *   Along each side of the square, there is one big circle center and one little circle center (distance between centers = side length = $a$).

**2. Formulate Equations**
*   **Condition 1: "The two big circles touch each other"**
    The distance between the centers of the two big circles is the diagonal of the square. Since they touch externally, the sum of their radii equals this distance.
    $$R + R = a\sqrt{2}$$
    $$2R = a\sqrt{2}$$
    $$a = \frac{2R}{\sqrt{2}} = R\sqrt{2}$$

*   **Condition 2: "...and also the two little circles"**
    This implies that the big circles touch the adjacent little circles. The distance between the center of a big circle and the center of an adjacent little circle is the side of the square, $a$. Since they touch, the sum of their radii equals this distance.
    $$R + r = a$$

**3. Solve for the Ratio**
Substitute the expression for $a$ from the first equation into the second equation:
$$R + r = R\sqrt{2}$$

We want to find the factor $k$ such that $R = k \cdot r$. So, let's solve for $R$ in terms of $r$.
$$r = R\sqrt{2} - R$$
$$r = R(\sqrt{2} - 1)$$
$$R = \frac{r}{\sqrt{2} - 1}$$

To simplify the fraction, rationalize the denominator by multiplying the numerator and denominator by $(\sqrt{2} + 1)$:
$$R = r \cdot \frac{1}{\sqrt{2} - 1} \cdot \frac{\sqrt{2} + 1}{\sqrt{2} + 1}$$
$$R = r \cdot \frac{\sqrt{2} + 1}{2 - 1}$$
$$R = r \cdot (\sqrt{2} + 1)$$
$$R = r \cdot (1 + \sqrt{2})$$

So, the factor is $1 + \sqrt{2}$.

**4. Conclusion**
Comparing this result with the given choices:
(A) $\frac{2}{9}$
(B) $\sqrt{5}$
(C) $0.8 \cdot \pi$
(D) 2.5
(E) $1+\sqrt{2}$

The correct choice is (E).

**Correct Answer:** (E)

-----------------------
{'prompt_tokens': 272, 'total_tokens': 7215, 'completion_tokens': 6943, 'prompt_tokens_details': None}


---
#### Image understanding

<img src="https://qianwen-res.oss-accelerate.aliyuncs.com/Qwen3.5/demo/RealWorld/RealWorld-04.png" width="800">

In [13]:
payload = {
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": "https://qianwen-res.oss-accelerate.aliyuncs.com/Qwen3.5/demo/RealWorld/RealWorld-04.png"
                    }
                },
                {
                    "type": "text",
                    "text": "Where is this?"
                }
            ]
        }
    ],
}

start_time = time.time()
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                 Body=json.dumps(payload),
                                 ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
end_time = time.time()

print(f"✅ Response time: {end_time-start_time:.2f}s\n")
display(Markdown(response["choices"][0]["message"]["content"]))

usage = response["usage"] 
print(f'-----------------------\n{usage}')

✅ Response time: 25.64s





Based on the visual evidence, this photo was taken in **Medellín, Colombia**.

Here are the specific details:

*   **The Venue:** The sign in the foreground reads **"Origen"** (specifically **@elorigen** or similar, partially visible as "@rigen"). This is a restaurant and bar called **El Origen**, located in the **Manrique** neighborhood of Medellín. It is known for this large sculpture and its panoramic views.
*   **The Sculpture:** The large statue is a bust of an indigenous figure with a golden headband, which serves as a decorative centerpiece for the venue's terrace.
*   **The Background:** The cityscape behind the statue is characteristic of Medellín. You can see the dense, red-brick houses climbing up the steep mountainsides (known as *comunas*), which is the iconic look of the city. The tall apartment blocks in the distance are likely part of the social housing projects or newer developments in the northern parts of the city.

-----------------------
{'prompt_tokens': 2468, 'total_tokens': 5157, 'completion_tokens': 2689, 'prompt_tokens_details': None}


---
### Performance improvement by using speculative decoding

#### By using Qwen3.5 native MTP, with an input size of 500 tokens and an output size of 500 tokens, the latency decreased by 40% and total token throughput increased by 41%.

#### With Speculative decoding

| Metric | avg | p25 | p50 | p75 | p90 | p95 | p99 |
|--------|-----|-----|-----|-----|-----|-----|-----|
| Input Sequence Length (tokens) | 500.72 | 472.5 | 501 | 526.5 | 576.1 | 586 | 612.08 |
| Inter Token Latency (ms) | 7.47 | 7.3 | 7.46 | 7.66 | 7.84 | 7.98 | 8.19 |
| Output Sequence Length (tokens) | 493.78 | 460 | 493 | 528.25 | 554.1 | 586.35 | 642.26 |
| Output Token Throughput Per User (tokens/sec/user) | 134.13 | 130.51 | 133.97 | 136.97 | 140.2 | 142.73 | 147.63 |
| Prefill Throughput Per User (tokens/sec/user) | 3445.28 | 3273.33 | 3493.09 | 3649.1 | 3950.52 | 4055.34 | 4289.4 |
| Reasoning Token Count (tokens) | 493.78 | 460 | 493 | 528.25 | 554.1 | 586.35 | 642.26 |
| Request Latency (ms) | 3838.25 | 3528.19 | 3774.99 | 4106.22 | 4412.84 | 4610.55 | 5237.05 |
| Time to First Token (ms) | 155.17 | 142.6 | 143.75 | 144.55 | 146.46 | 147.42 | 190.47 |

#### No Speculative decoding

| Metric | avg | p25 | p50 | p75 | p90 | p95 | p99 |
|--------|-----|-----|-----|-----|-----|-----|-----|
| Input Sequence Length (tokens) | 500.72 | 472.5 | 501 | 526.5 | 576.1 | 586 | 612.08 |
| Inter Token Latency (ms) | 12.68 | 12.58 | 12.67 | 12.77 | 12.91 | 12.98 | 13.1 |
| Output Sequence Length (tokens) | 495.04 | 461.5 | 493 | 526.75 | 556.1 | 578.9 | 649.23 |
| Output Token Throughput Per User (tokens/sec/user) | 78.9 | 78.29 | 78.91 | 79.47 | 80.25 | 80.7 | 81.76 |
| Prefill Throughput Per User (tokens/sec/user) | 3440.07 | 3293.77 | 3500.32 | 3704.06 | 4006.08 | 4076.01 | 4331.11 |
| Reasoning Token Count (tokens) | 495.04 | 461.5 | 493 | 526.75 | 556.1 | 578.9 | 649.23 |
| Request Latency (ms) | 6533.25 | 6000.32 | 6374.16 | 6883.66 | 7209.45 | 7647.59 | 8849.3 |
| Time to First Token (ms) | 271.08 | 141.47 | 142.18 | 143.35 | 144.75 | 149.07 | 598.6 |

## Cleanup

In [14]:
_ = sm.delete_endpoint(EndpointName=endpoint_name)
_ = sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
_ = sm.delete_model(ModelName=model_name)